# Homework 2 – Web scraping: Aviation

**Goal:** scrape the supplied local website with Python and Beautiful Soup and export the 8 data groups to CSV files.

## Setup
- Python 3 (Anaconda) and Jupyter Notebook (local)
- Packages: `beautifulsoup4`, `pandas`
- Folder layout: this notebook must sit next to the extracted `website/` folder:
  `homework2.ipynb` + `website/index.html` + `website/<data group folders>`.
- The HTML files are read directly from disk with `Path.read_text(encoding="utf-8")`, so no web server or `requests.get()` is needed.
- Run all cells from top to bottom. The full notebook runs in about 15 minutes.
- Rerunning the notebook overwrites the CSV files (`to_csv`), so no duplicate rows are appended.

## 1. Exploring the website
I first opened `index.html` in the browser and used the Inspect tool, then counted the HTML pages per folder.
- There is one folder per data group (aircrafts_data, airports_data, seats, flights, bookings, tickets, ticket_flights, boarding_passes).
- Every page stores its records in an HTML table: `table > tr` (one row per record) `> td` (one cell per field). The first `tr` holds the column headers (`th`).
- Comparing the number of pages with the row counts of the assignment shows about **250 rows per page**. Page 1 is `index.html`, then `page-2.html`, `page-3.html`, etc.

In [1]:
from pathlib import Path        
from bs4 import BeautifulSoup    
import pandas as pd            

website = Path("website")                 
print((website / "index.html").is_file())

True


In [2]:
groups = ["aircrafts_data", "airports_data", "seats", "flights",
          "bookings", "tickets", "ticket_flights", "boarding_passes"]

for g in groups:
    files = list((website / g).glob("*.html"))   # all HTML pages of this folder
    print(g, len(files))

aircrafts_data 1
airports_data 1
seats 6
flights 133
bookings 1052
tickets 1467
ticket_flights 4183
boarding_passes 2319


## 2. Aircrafts (Fleet) – 1 page
Method : inspect one element first (`prettify()`), extract one row, then generalise with a `for` loop over the rows and build a DataFrame from lists.
- The model is multilingual, for example `Boeing 777-300 / Боинг 777-300`. I split the visible text on `" / "` into two columns, `model_en` and `model_ru`, so each language can be used separately.
- `range` is converted to an integer.
- The loop uses `rows[1:]` because `rows[0]` is the header row, which contains `th` cells and no `td`.

In [3]:
fleet_page = website / "aircrafts_data" / "index.html"
html = fleet_page.read_text(encoding="utf-8")
soup = BeautifulSoup(html, "html.parser")

table = soup.find("table")
rows = table.find_all("tr")
print("row length <tr> :", len(rows))

print(rows[0].prettify())
print(rows[1].prettify())

row length <tr> : 10
<tr>
 <th data-field="aircraft_code" scope="col">
  Aircraft
 </th>
 <th data-field="model" scope="col">
  Model
 </th>
 <th data-field="range" scope="col">
  Range · km
 </th>
</tr>

<tr data-record="aircrafts_data" id="r1">
 <td>
  773
 </td>
 <td data-value='{"en": "Boeing 777-300", "ru": "Боинг 777-300"}'>
  Boeing 777-300 / Боинг 777-300
 </td>
 <td>
  11100
 </td>
</tr>



In [4]:
row = rows[1]                        
cells = row.find_all("td")         

aircraft_code = cells[0].text.strip()      
model = cells[1].text.strip()              
model_en = model.split(" / ")[0]            
model_ru = model.split(" / ")[1]            
range_km = int(cells[2].text.strip())      

print(aircraft_code, "|", model_en, "|", model_ru, "|", range_km)

773 | Boeing 777-300 | Боинг 777-300 | 11100


In [5]:
code_list = []
model_en_list = []
model_ru_list = []
range_list = []

for row in rows[1:]:                       
    cells = row.find_all("td")              

    aircraft_code = cells[0].text.strip()
    model = cells[1].text.strip()
    model_en = model.split(" / ")[0]
    model_ru = model.split(" / ")[1]
    range_km = int(cells[2].text.strip())

    code_list.append(aircraft_code)
    model_en_list.append(model_en)
    model_ru_list.append(model_ru)
    range_list.append(range_km)

fleet = pd.DataFrame({
    "aircraft_code": code_list,
    "model_en": model_en_list,
    "model_ru": model_ru_list,
    "range": range_list,
})
print(fleet.shape)
fleet

(9, 4)


,aircraft_code,model_en,model_ru,range
0,773,Boeing 777-300,Боинг 777-300,11100
1,763,Boeing 767-300,Боинг 767-300,7900
2,SU9,Sukhoi Superjet-100,Сухой Суперджет-100,3000
3,320,Airbus A320-200,Аэробус A320-200,5700
4,321,Airbus A321-200,Аэробус A321-200,5600
5,319,Airbus A319-100,Аэробус A319-100,6700
6,733,Boeing 737-300,Боинг 737-300,4200
7,CN1,Cessna 208 Caravan,Сессна 208 Караван,1200
8,CR2,Bombardier CRJ-200,Бомбардье CRJ-200,2700


In [6]:
fleet.to_csv("aircrafts_data.csv", index=False, encoding="utf-8-sig")

In [7]:
groups = ["airports_data", "seats", "flights", "bookings",
          "tickets", "ticket_flights", "boarding_passes"]

for g in groups:
    page = website / g / "index.html"              
    soup = BeautifulSoup(page.read_text(encoding="utf-8"), "html.parser")
    rows = soup.find("table").find_all("tr")
    print("=====", g, "- rows on page 1:", len(rows))
    print(rows[0].prettify())                        
    print(rows[1].prettify())                         

===== airports_data - rows on page 1: 105
<tr>
 <th data-field="airport_code" scope="col">
  IATA
 </th>
 <th data-field="airport_name" scope="col">
  Airport
 </th>
 <th data-field="city" scope="col">
  City
 </th>
 <th data-field="coordinates" scope="col">
  Coordinates
 </th>
 <th data-field="timezone" scope="col">
  Time zone
 </th>
</tr>

<tr data-record="airports_data" id="r1">
 <td>
  YKS
 </td>
 <td data-value='{"en": "Yakutsk Airport", "ru": "Якутск"}'>
  Yakutsk Airport / Якутск
 </td>
 <td data-value='{"en": "Yakutsk", "ru": "Якутск"}'>
  Yakutsk / Якутск
 </td>
 <td>
  (129.77099609375,62.0932998657226562)
 </td>
 <td>
  Asia/Yakutsk
 </td>
</tr>

===== seats - rows on page 1: 251
<tr>
 <th data-field="aircraft_code" scope="col">
  Aircraft
 </th>
 <th data-field="seat_no" scope="col">
  Seat
 </th>
 <th data-field="fare_conditions" scope="col">
  Cabin
 </th>
</tr>

<tr data-record="seats" id="r1">
 <td>
  <a href="../aircrafts_data/index.html#r6">
   319
  </a>
 </td>
 <t

In [8]:
airport_page = website / "airports_data" / "index.html"
html = airport_page.read_text(encoding="utf-8")
soup = BeautifulSoup(html, "html.parser")

table = soup.find("table")
rows = table.find_all("tr")
print("row length <tr> :", len(rows))

print(rows[0].prettify())
print(rows[1].prettify())

row length <tr> : 105
<tr>
 <th data-field="airport_code" scope="col">
  IATA
 </th>
 <th data-field="airport_name" scope="col">
  Airport
 </th>
 <th data-field="city" scope="col">
  City
 </th>
 <th data-field="coordinates" scope="col">
  Coordinates
 </th>
 <th data-field="timezone" scope="col">
  Time zone
 </th>
</tr>

<tr data-record="airports_data" id="r1">
 <td>
  YKS
 </td>
 <td data-value='{"en": "Yakutsk Airport", "ru": "Якутск"}'>
  Yakutsk Airport / Якутск
 </td>
 <td data-value='{"en": "Yakutsk", "ru": "Якутск"}'>
  Yakutsk / Якутск
 </td>
 <td>
  (129.77099609375,62.0932998657226562)
 </td>
 <td>
  Asia/Yakutsk
 </td>
</tr>



In [9]:
row = rows[1]                         
cells = row.find_all("td")            

airport_code = cells[0].text.strip()                  

airport_name = cells[1].text.strip()                  
airport_name_en = airport_name.split(" / ")[0]            
airport_name_ru = airport_name.split(" / ")[1]             

city = cells[2].text.strip()                           
city_en = city.split(" / ")[0]
city_ru = city.split(" / ")[1]

coordinates = cells[3].text.strip()                      
coordinates = coordinates.replace("(", "").replace(")", "")   
longitude = float(coordinates.split(",")[0])
latitude = float(coordinates.split(",")[1])             

timezone = cells[4].text.strip()                      

print(airport_code, airport_name_en, airport_name_ru, city_en, city_ru, longitude, latitude, timezone)

YKS Yakutsk Airport Якутск Yakutsk Якутск 129.77099609375 62.093299865722656 Asia/Yakutsk


## 3. Airports – 1 page
- `airport_name` and `city` are multilingual: they are split into `_en` and `_ru` columns, like the aircraft model.
- `coordinates` is stored as `(longitude,latitude)`, for example `(129.77,62.09)` for Yakutsk (129.77° E, 62.09° N). I removed the brackets and split on the comma to get two numeric columns, `longitude` and `latitude`.
- `timezone` is kept as text (for example `Asia/Yakutsk`).

In [10]:
code_list = []
name_en_list = []
name_ru_list = []
city_en_list = []
city_ru_list = []
longitude_list = []
latitude_list = []
timezone_list = []

for row in rows[1:]:                               
    cells = row.find_all("td")                     

    airport_code = cells[0].text.strip()          

    airport_name = cells[1].text.strip()
    airport_name_en = airport_name.split(" / ")[0]
    airport_name_ru = airport_name.split(" / ")[1]

    city = cells[2].text.strip()
    city_en = city.split(" / ")[0]
    city_ru = city.split(" / ")[1]

    coordinates = cells[3].text.strip()
    coordinates = coordinates.replace("(", "").replace(")", "")
    longitude = float(coordinates.split(",")[0])
    latitude = float(coordinates.split(",")[1])
    timezone = cells[4].text.strip()             

    code_list.append(airport_code)
    name_en_list.append(airport_name_en)
    name_ru_list.append(airport_name_ru)
    city_en_list.append(city_en)
    city_ru_list.append(city_ru)
    longitude_list.append(longitude)
    latitude_list.append(latitude)
    timezone_list.append(timezone)

airports = pd.DataFrame({
    "airport_code": code_list,
    "airport_name_en": name_en_list,
    "airport_name_ru": name_ru_list,
    "city_en": city_en_list,
    "city_ru": city_ru_list,
    "longitude": longitude_list,
    "latitude": latitude_list,
    "timezone": timezone_list,
})
print(airports.shape)  
airports.head()

(104, 8)


,airport_code,airport_name_en,airport_name_ru,city_en,city_ru,longitude,latitude,timezone
0,YKS,Yakutsk Airport,Якутск,Yakutsk,Якутск,129.770996,62.093300,Asia/Yakutsk
1,MJZ,Mirny Airport,Мирный,Mirnyj,Мирный,114.039001,62.534698,Asia/Yakutsk
2,KHV,Khabarovsk-Novy Airport,Хабаровск-Новый,Khabarovsk,Хабаровск,135.188004,48.528000,Asia/Vladivostok
3,PKC,Yelizovo Airport,Елизово,Petropavlovsk,Петропавловск-Камчатский,158.453995,53.167900,Asia/Kamchatka
4,UUS,Yuzhno-Sakhalinsk Airport,Хомутово,Yuzhno-Sakhalinsk,Южно-Сахалинск,142.718002,46.888699,Asia/Sakhalin


In [11]:
airports.to_csv("airports_data.csv", index=False, encoding="utf-8-sig") 

## 4. Pagination (Seats, 6 pages)
The same pattern as `books_to_scrape` : for each page, extract its rows into lists, build a small DataFrame, add it to `data_list`, then combine everything with `pd.concat`.
- Page file names are built in numeric order (`index.html`, then `"page-" + str(i) + ".html"`). Sorting the file names alphabetically would put `page-10` before `page-2`.
- `rows` is recomputed from each new page inside the loop.

In [12]:
folder = website / "seats"
nb_pages = len(list(folder.glob("*.html")))
print("Pages :", nb_pages)

data_list = []                                  

for i in range(1, nb_pages + 1):                
    if i == 1:
        page = folder / "index.html"            
    else:
        page = folder / ("page-" + str(i) + ".html")

    soup = BeautifulSoup(page.read_text(encoding="utf-8"), "html.parser")
    rows = soup.find("table").find_all("tr")

    aircraft_list = []
    seat_list = []
    fare_list = []

    for row in rows[1:]:
        cells = row.find_all("td")
        aircraft_list.append(cells[0].text.strip())   
        seat_list.append(cells[1].text.strip())      
        fare_list.append(cells[2].text.strip())       

    data = pd.DataFrame({"aircraft_code": aircraft_list,
                         "seat_no": seat_list,
                         "fare_conditions": fare_list})
    data_list.append(data)

seats = pd.concat(data_list)
print(seats.shape)        
seats.head()

Pages : 6
(1339, 3)


,aircraft_code,seat_no,fare_conditions
0,319,2A,Business
1,319,2C,Business
2,319,2D,Business
3,319,2F,Business
4,319,3A,Business


In [13]:
seats.to_csv("seats.csv", index=False) 

In [14]:
print(seats.shape)
print(airports.shape)

(1339, 3)
(104, 8)


## 5. Flights (133 pages)
- **Missing values:** for flights that have not departed yet, the page displays `Pending`, but the cell attribute is `data-value="\N"`, the database marker for a missing value. `Pending` is not a timestamp, so these cells are stored as empty values (`None`). In Python the marker is written `"\\N"`, because `\` is an escape character.
- Timestamps such as `2017-09-10 09:50:00+03` are kept as text, so that the time zone offset (`+03`) is preserved exactly as published.
- Airport and aircraft codes are inside links (`<a>`); `.text` returns the visible code.

In [15]:
import time
start = time.time()                              

folder = website / "flights"
nb_pages = len(list(folder.glob("*.html")))        
data_list = []

for i in range(1, nb_pages + 1):
    if i == 1:
        page = folder / "index.html"
    else:
        page = folder / ("page-" + str(i) + ".html")

    soup = BeautifulSoup(page.read_text(encoding="utf-8"), "html.parser")
    rows = soup.find("table").find_all("tr")

    id_list, no_list, sched_dep_list, sched_arr_list = [], [], [], []
    dep_list, arr_list, status_list, aircraft_list = [], [], [], []
    act_dep_list, act_arr_list = [], []

    for row in rows[1:]:
        cells = row.find_all("td")
        id_list.append(cells[0].text.strip())
        no_list.append(cells[1].text.strip())
        sched_dep_list.append(cells[2].text.strip())       
        sched_arr_list.append(cells[3].text.strip())
        dep_list.append(cells[4].text.strip())           
        arr_list.append(cells[5].text.strip())
        status_list.append(cells[6].text.strip())
        aircraft_list.append(cells[7].text.strip())

        if cells[8].get("data-value") == "\\N":
            act_dep_list.append(None)
        else:
            act_dep_list.append(cells[8].text.strip())

        if cells[9].get("data-value") == "\\N":
            act_arr_list.append(None)
        else:
            act_arr_list.append(cells[9].text.strip())

    data = pd.DataFrame({
        "flight_id": id_list, "flight_no": no_list,
        "scheduled_departure": sched_dep_list, "scheduled_arrival": sched_arr_list,
        "departure_airport": dep_list, "arrival_airport": arr_list,
        "status": status_list, "aircraft_code": aircraft_list,
        "actual_departure": act_dep_list, "actual_arrival": act_arr_list,
    })
    data_list.append(data)

flights = pd.concat(data_list)
print(flights.shape)                                        
print("missing departure :", flights["actual_departure"].isna().sum())   
print("duration :", round(time.time() - start), "seconds")   
flights.to_csv("flights.csv", index=False)

(33121, 10)
missing departure : 16348
duration : 18 seconds


## 6. Bookings, Tickets, Ticket flights, Boarding passes
Same pagination pattern. Specific choices:
- **Amounts** (`total_amount`, `amount`) are read from the `data-value` attribute (`265700`) rather than the displayed text (`265,700.00`), which contains a thousands separator.
- **Identifiers** (`ticket_no`, `book_ref`, `flight_id`) are kept as text: `ticket_no` values start with zeros (`0005432000987`), and converting them to numbers would remove those zeros.
- These are the largest groups (up to 4,183 pages), and they took most of the run time.

In [16]:
start = time.time()                                  

folder = website / "bookings"
nb_pages = len(list(folder.glob("*.html")))        
data_list = []                                        

for i in range(1, nb_pages + 1):
    if i % 500 == 0:                                  
        print("Page", i, "of", nb_pages)
    if i == 1:
        page = folder / "index.html"                 
    else:
        page = folder / ("page-" + str(i) + ".html") 

    soup = BeautifulSoup(page.read_text(encoding="utf-8"), "html.parser")
    rows = soup.find("table").find_all("tr")        

    ref_list = []
    date_list = []
    amount_list = []

    for row in rows[1:]:                              
        cells = row.find_all("td")
        ref_list.append(cells[0].text.strip())                
        date_list.append(cells[1].text.strip())                
        amount_list.append(float(cells[2].get("data-value"))) 

    data = pd.DataFrame({"book_ref": ref_list,
                         "book_date": date_list,
                         "total_amount": amount_list})
    data_list.append(data)

bookings = pd.concat(data_list)
print(bookings.shape)                                 
print("duration :", round(time.time() - start), "seconds")
bookings.to_csv("bookings.csv", index=False)

Page 500 of 1052
Page 1000 of 1052
(262788, 3)
duration : 52 seconds


In [17]:
start = time.time()

folder = website / "tickets"
nb_pages = len(list(folder.glob("*.html")))          
data_list = []

for i in range(1, nb_pages + 1):
    if i % 500 == 0:
        print("Page", i, "of", nb_pages)
    if i == 1:
        page = folder / "index.html"
    else:
        page = folder / ("page-" + str(i) + ".html")

    soup = BeautifulSoup(page.read_text(encoding="utf-8"), "html.parser")
    rows = soup.find("table").find_all("tr")

    ticket_list = []
    ref_list = []
    passenger_list = []

    for row in rows[1:]:
        cells = row.find_all("td")
        ticket_list.append(cells[0].text.strip())     
        ref_list.append(cells[1].text.strip())        
        passenger_list.append(cells[2].text.strip()) 

    data = pd.DataFrame({"ticket_no": ticket_list,
                         "book_ref": ref_list,
                         "passenger_id": passenger_list})
    data_list.append(data)

tickets = pd.concat(data_list)
print(tickets.shape)                                  
print("duration :", round(time.time() - start), "seconds")
tickets.to_csv("tickets.csv", index=False)

Page 500 of 1467
Page 1000 of 1467
(366733, 3)
duration : 89 seconds


In [18]:
start = time.time()

folder = website / "ticket_flights"
nb_pages = len(list(folder.glob("*.html")))          
data_list = []

for i in range(1, nb_pages + 1):
    if i % 500 == 0:
        print("Page", i, "of", nb_pages)
    if i == 1:
        page = folder / "index.html"
    else:
        page = folder / ("page-" + str(i) + ".html")

    soup = BeautifulSoup(page.read_text(encoding="utf-8"), "html.parser")
    rows = soup.find("table").find_all("tr")

    ticket_list = []
    flight_list = []
    fare_list = []
    amount_list = []

    for row in rows[1:]:
        cells = row.find_all("td")
        ticket_list.append(cells[0].text.strip())            
        flight_list.append(cells[1].text.strip())             
        fare_list.append(cells[2].text.strip())               
        amount_list.append(float(cells[3].get("data-value")))

    data = pd.DataFrame({"ticket_no": ticket_list,
                         "flight_id": flight_list,
                         "fare_conditions": fare_list,
                         "amount": amount_list})
    data_list.append(data)

ticket_flights = pd.concat(data_list)
print(ticket_flights.shape)                           
print("duration :", round(time.time() - start), "seconds")
ticket_flights.to_csv("ticket_flights.csv", index=False)

Page 500 of 4183
Page 1000 of 4183
Page 1500 of 4183
Page 2000 of 4183
Page 2500 of 4183
Page 3000 of 4183
Page 3500 of 4183
Page 4000 of 4183
(1045726, 4)
duration : 334 seconds


In [19]:
start = time.time()

folder = website / "boarding_passes"
nb_pages = len(list(folder.glob("*.html")))
data_list = []

for i in range(1, nb_pages + 1):
    if i % 500 == 0:
        print("Page", i, "of", nb_pages)
    if i == 1:
        page = folder / "index.html"
    else:
        page = folder / ("page-" + str(i) + ".html")

    soup = BeautifulSoup(page.read_text(encoding="utf-8"), "html.parser")
    rows = soup.find("table").find_all("tr")

    ticket_list = []
    flight_list = []
    boarding_list = []
    seat_list = []

    for row in rows[1:]:
        cells = row.find_all("td")
        ticket_list.append(cells[0].text.strip())         
        flight_list.append(cells[1].text.strip())         
        boarding_list.append(int(cells[2].text.strip()))  
        seat_list.append(cells[3].text.strip())           

    data = pd.DataFrame({"ticket_no": ticket_list,
                         "flight_id": flight_list,
                         "boarding_no": boarding_list,
                         "seat_no": seat_list})
    data_list.append(data)

boarding_passes = pd.concat(data_list)
print(boarding_passes.shape)                         
print("duration :", round(time.time() - start), "seconds")
boarding_passes.to_csv("boarding_passes.csv", index=False)

Page 500 of 2319
Page 1000 of 2319
Page 1500 of 2319
Page 2000 of 2319
(579686, 4)
duration : 183 seconds


In [20]:
print(bookings.shape)                                 
print(tickets.shape)                                  
print(ticket_flights.shape)                           
print(boarding_passes.shape)                          

(262788, 3)
(366733, 3)
(1045726, 4)
(579686, 4)


## 7. CSV structure
I kept **one CSV file per data group (8 files)**, as recommended, because each group has its own key and the files can be joined later through the relationships listed in the assignment.

| Data group | CSV file | Columns | Change vs. reference |
|---|---|---|---|
| aircrafts_data | aircrafts_data.csv | aircraft_code, model_en, model_ru, range | `model` split by language |
| airports_data | airports_data.csv | airport_code, airport_name_en/_ru, city_en/_ru, longitude, latitude, timezone | names split by language, coordinates split |
| seats | seats.csv | aircraft_code, seat_no, fare_conditions | none |
| flights | flights.csv | 10 fields | none |
| bookings | bookings.csv | book_ref, book_date, total_amount | none |
| tickets | tickets.csv | ticket_no, book_ref, passenger_id | none |
| ticket_flights | ticket_flights.csv | ticket_no, flight_id, fare_conditions, amount | none |
| boarding_passes | boarding_passes.csv | ticket_no, flight_id, boarding_no, seat_no | none |

The two files that contain Cyrillic text are saved with `utf-8-sig` so that Excel displays them correctly.

## 8. Checks

## check 1 

In [21]:
reference = {"aircrafts_data": 9, "airports_data": 104, "seats": 1339,
             "flights": 33121, "bookings": 262788, "tickets": 366733,
             "ticket_flights": 1045726, "boarding_passes": 579686}

for name, expected in reference.items():
    df = pd.read_csv(name + ".csv", dtype=str)     
    print(name, "| CSV :", len(df), "| expected :", expected, "| OK :", len(df) == expected)

aircrafts_data | CSV : 9 | expected : 9 | OK : True
airports_data | CSV : 104 | expected : 104 | OK : True
seats | CSV : 1339 | expected : 1339 | OK : True
flights | CSV : 33121 | expected : 33121 | OK : True
bookings | CSV : 262788 | expected : 262788 | OK : True
tickets | CSV : 366733 | expected : 366733 | OK : True
ticket_flights | CSV : 1045726 | expected : 1045726 | OK : True
boarding_passes | CSV : 579686 | expected : 579686 | OK : True


## check 2 

In [22]:
print("flights         :", flights.duplicated(subset=["flight_id"]).sum())
print("bookings        :", bookings.duplicated(subset=["book_ref"]).sum())
print("tickets         :", tickets.duplicated(subset=["ticket_no"]).sum())
print("ticket_flights  :", ticket_flights.duplicated(subset=["ticket_no", "flight_id"]).sum())
print("boarding_passes :", boarding_passes.duplicated(subset=["ticket_no", "flight_id"]).sum())
print("seats           :", seats.duplicated(subset=["aircraft_code", "seat_no"]).sum())
print("airports        :", airports.duplicated(subset=["airport_code"]).sum())
print("aircrafts       :", fleet.duplicated(subset=["aircraft_code"]).sum())

flights         : 0
bookings        : 0
tickets         : 0
ticket_flights  : 0
boarding_passes : 0
seats           : 0
airports        : 0
aircrafts       : 0


## check 3 

In [23]:
print("tickets.book_ref -> bookings        :", tickets["book_ref"].isin(bookings["book_ref"]).all())
print("ticket_flights.ticket_no -> tickets :", ticket_flights["ticket_no"].isin(tickets["ticket_no"]).all())
print("ticket_flights.flight_id -> flights :", ticket_flights["flight_id"].isin(flights["flight_id"]).all())
print("boarding.ticket_no -> tickets       :", boarding_passes["ticket_no"].isin(tickets["ticket_no"]).all())
print("boarding.flight_id -> flights       :", boarding_passes["flight_id"].isin(flights["flight_id"]).all())
print("flights.aircraft_code -> aircrafts  :", flights["aircraft_code"].isin(fleet["aircraft_code"]).all())
print("seats.aircraft_code -> aircrafts    :", seats["aircraft_code"].isin(fleet["aircraft_code"]).all())
print("flights.departure -> airports       :", flights["departure_airport"].isin(airports["airport_code"]).all())
print("flights.arrival -> airports         :", flights["arrival_airport"].isin(airports["airport_code"]).all())

tickets.book_ref -> bookings        : True
ticket_flights.ticket_no -> tickets : True
ticket_flights.flight_id -> flights : True
boarding.ticket_no -> tickets       : True
boarding.flight_id -> flights       : True
flights.aircraft_code -> aircrafts  : True
seats.aircraft_code -> aircrafts    : True
flights.departure -> airports       : True
flights.arrival -> airports         : True


## check 4 

In [24]:
print(flights.isna().sum())          
print(flights[flights["actual_departure"].isna()]["status"].value_counts())

flight_id                  0
flight_no                  0
scheduled_departure        0
scheduled_arrival          0
departure_airport          0
arrival_airport            0
status                     0
aircraft_code              0
actual_departure       16348
actual_arrival         16406
dtype: int64
status
Scheduled    15383
On Time        518
Cancelled      406
Delayed         41
Name: count, dtype: int64


## check 5 

In [25]:
ticket_flights.tail(3)            

,ticket_no,flight_id,fare_conditions,amount
223,0005435104384,32094,Economy,5200.0
224,0005435104352,32094,Economy,5200.0
225,0005435104389,32094,Economy,5200.0


In [26]:
pd.read_csv("aircrafts_data.csv")   

,aircraft_code,model_en,model_ru,range
0,773,Boeing 777-300,Боинг 777-300,11100
1,763,Boeing 767-300,Боинг 767-300,7900
2,SU9,Sukhoi Superjet-100,Сухой Суперджет-100,3000
3,320,Airbus A320-200,Аэробус A320-200,5700
4,321,Airbus A321-200,Аэробус A321-200,5600
5,319,Airbus A319-100,Аэробус A319-100,6700
6,733,Boeing 737-300,Боинг 737-300,4200
7,CN1,Cessna 208 Caravan,Сессна 208 Караван,1200
8,CR2,Bombardier CRJ-200,Бомбардье CRJ-200,2700


In [27]:
ticket = boarding_passes["ticket_no"].iloc[0]    
flight = boarding_passes["flight_id"].iloc[0]     

print(tickets[tickets["ticket_no"] == ticket])                
book = tickets[tickets["ticket_no"] == ticket]["book_ref"].iloc[0]
print(bookings[bookings["book_ref"] == book])                 
print(ticket_flights[ticket_flights["ticket_no"] == ticket])   
print(flights[flights["flight_id"] == flight])                 

        ticket_no book_ref passenger_id
68  0005435212351   BEAA38  9933 118369
    book_ref               book_date  total_amount
189   BEAA38  2017-06-29 08:47:00+03      112200.0
        ticket_no flight_id fare_conditions   amount
1   0005435212351     30625        Business  42100.0
97  0005435212351      7744        Business  42100.0
    flight_id flight_no     scheduled_departure       scheduled_arrival  \
125     30625    PG0013  2017-07-16 18:15:00+03  2017-07-16 20:00:00+03   

    departure_airport arrival_airport   status aircraft_code  \
125               AER             SVO  Arrived           773   

           actual_departure          actual_arrival  
125  2017-07-16 18:18:00+03  2017-07-16 20:04:00+03  


**Check results**
- **Row counts:** all 8 CSV files, read back from disk, have exactly the reference number of rows. This also shows that rerunning the export did not append duplicates.
- **Duplicates:** 0 duplicated keys in every group (e.g. `ticket_no + flight_id` for ticket_flights). Each record appears once.
- **Relationships:** every foreign key exists in its parent table (tickets → bookings, ticket_flights/boarding_passes → tickets and flights, flights/seats → aircrafts, flights → airports).
- **Missing values:** 16,348 flights have no actual departure. Their statuses are all statuses of flights that have not departed, which is consistent.
- **Later pages:** the last rows of `ticket_flights` were compared manually with `page-4183.html` and are identical.

## 9. Unresolved issues and limitations
- Timestamps are stored as text and not converted to a datetime type. This would be done during the analysis.
- The multilingual split relies on the `" / "` separator; a name containing this separator would be split incorrectly.
- When reading the CSV files back, use `dtype=str` for identifiers to keep their leading zeros.
- On a French-locale Excel, CSV files open in a single column because Excel expects `;` as the separator. The files themselves use the standard `,` separator.